# 71 — P10.8: auditoría Sudirman / Al-Kafri

Audita fuentes reales para facetas, ligamento amarillo, raíces, grasa epidural, altura discal, diámetros, listesis y hernia multiframe. **No entrena, no carga `.pt`, no abre tests sellados y no crea ground truth.** Sudirman y Al-Kafri se consideran una misma familia de cohorte cuando comparten pacientes.

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Entorno no Colab')

Mounted at /content/drive


In [13]:
from __future__ import annotations
import hashlib,json,os,re,xml.etree.ElementTree as ET
from collections import Counter
from datetime import datetime,timezone
from pathlib import Path
import pandas as pd

ROOT=Path(os.getenv('PFI_ROOT','/content/drive/MyDrive/PFI_MVP'))
PREF=Path(os.getenv('PFI_P10_8_PREFLIGHT_ROOT',str(ROOT/'results/P10_8_clinical_expansion_preflight')))
OUT=Path(os.getenv('PFI_P10_8_SUDIRMAN_AUDIT_ROOT',str(PREF/'sudirman_alkafri_audit')))
for m in [PREF/'NOTEBOOK_69_COMPLETE.json',PREF/'NOTEBOOK_70_COMPLETE.json']:
    if not m.is_file(): raise FileNotFoundError(f'Falta marcador previo: {m}')
MAX_FILES=int(os.getenv('PFI_P10_8_MAX_FILES_TO_INDEX','250000'))
MAX_TEXT=int(os.getenv('PFI_P10_8_MAX_TEXT_FILES_TO_SCAN','5000'))
MAX_BYTES=int(os.getenv('PFI_P10_8_MAX_BYTES_PER_TEXT_FILE',str(4*1024*1024)))
MAX_ROWS=int(os.getenv('PFI_P10_8_MAX_TABULAR_ROWS_TO_SCAN','5000'))
sha=lambda s:hashlib.sha256(str(s).encode()).hexdigest()
norm=lambda s:re.sub(r'[^a-z0-9]+','_',str(s).lower()).strip('_')
def write_json(p,x):
    p.parent.mkdir(parents=True,exist_ok=True); t=p.with_suffix(p.suffix+'.tmp'); t.write_text(json.dumps(x,indent=2,ensure_ascii=False,sort_keys=True)+'\n',encoding='utf-8'); os.replace(t,p)
def uniq(xs):
    out=[]; seen=set()
    for x in xs:
        p=Path(x)
        if p.exists() and str(p.resolve()) not in seen: seen.add(str(p.resolve())); out.append(p)
    return out
manual=[]
for k in ['PFI_SUDIRMAN_ROOT','PFI_ALKAFRI_ROOT']:
    if os.getenv(k,'').strip(): manual.append(Path(os.getenv(k)))
manual += [Path(x.strip()) for x in re.split(r'[;\n]+',os.getenv('PFI_P10_8_DATA_ROOTS','')) if x.strip()]
specific=[ROOT/'data/Sudirman',ROOT/'data/sudirman',ROOT/'data/AlKafri',ROOT/'data/Al-Kafri',ROOT/'data/alkafri',ROOT/'data/Lumbar_MRI',ROOT/'data/LumbarMRI',ROOT/'results/E6_alkafri_axial_pairing',ROOT/'results/E7_alkafri_axial_curated_subset']
roots=uniq(manual+specific) or uniq([ROOT/'data',ROOT/'datasets',ROOT/'Dataset'])
root_df=pd.DataFrame([{'rootAlias':f'root_{i:02d}','rootPathHash':sha(p.resolve()),'rootName':p.name,'resolutionMethod':'configured' if p in manual else 'auto_detected'} for i,p in enumerate(roots,1)])
print('Raíces:',len(root_df)); display(root_df)

Raíces: 2


,rootAlias,rootPathHash,rootName,resolutionMethod
0,root_01,8f60e15fa96a149b445c4bff33ba4498a12d9bfac92a72...,_nested,configured
1,root_02,ab88a802a91c217d66c83c4f4b3b214dbaec707a6f6103...,E7_alkafri_axial_curated_subset,auto_detected


In [14]:
GROUPS={'dicom':{'.dcm','.ima','.dicom'},'volume':{'.nii','.nii.gz','.mha','.mhd','.nrrd'},'mask':{'.png','.bmp','.tif','.tiff'},'xml':{'.xml'},'table':{'.csv','.tsv','.xlsx','.xls','.parquet'},'structured':{'.json','.yaml','.yml'},'text':{'.txt','.md','.rtf'},'document':{'.pdf','.docx','.doc'},'model':{'.pt','.pth','.onnx'}}
def suff(p): return '.nii.gz' if p.name.lower().endswith('.nii.gz') else (p.suffix.lower() or '<none>')
def kind(p):
    s=suff(p)
    return next((g for g,v in GROUPS.items() if s in v),'other')
rows=[]; truncated=False
for i,r in enumerate(roots,1):
    for p in r.rglob('*'):
        if not p.is_file(): continue
        if len(rows)>=MAX_FILES: truncated=True; break
        rel=p.relative_to(r); rows.append({'rootAlias':f'root_{i:02d}','relativePathHash':sha(rel),'suffix':suff(p),'fileGroup':kind(p),'sizeBytes':p.stat().st_size,'_path':p})
    if truncated: break
private=pd.DataFrame(rows)
if private.empty: private=pd.DataFrame(columns=['rootAlias','relativePathHash','suffix','fileGroup','sizeBytes','_path'])
summary=private.groupby(['rootAlias','fileGroup','suffix'],dropna=False).agg(fileCount=('relativePathHash','count'),totalBytes=('sizeBytes','sum')).reset_index()
print('Archivos:',len(private),'truncado:',truncated); display(summary.head(100))

Archivos: 54593 truncado: False


,rootAlias,fileGroup,suffix,fileCount,totalBytes
0,root_01,dicom,.ima,17497,4891858350
1,root_01,mask,.png,29355,1057584724
2,root_01,other,.xcf,7725,799588173
3,root_02,structured,.json,4,1279
4,root_02,table,.csv,12,14768049


In [15]:
PAT={
'facet_hypertrophy':[r'facet',r'facetar',r'zygapophy'],
'ligamentum_flavum_hypertrophy':[r'ligamentum\s+flavum',r'ligamento\s+amarillo',r'flavum'],
'annular_tear':[r'annular\s+(tear|fissure)',r'(desgarro|fisura)\s+anular'],
'nerve_root_compression':[r'nerve\s+root',r'root\s+compression',r'radicular',r'ra[ií]z\s+nerv'],
'epidural_fat':[r'epidural\s+fat',r'grasa\s+epidural',r'fat\s+effacement'],
'disc_height':[r'(disc|disk)\s+height',r'altura\s+discal',r'height\s+loss'],
'ap_diameter':[r'anteroposterior',r'ap\s+diameter',r'di[aá]metro\s+ap'],
'spondylolisthesis':[r'spondylolisthesis',r'anterolisthesis',r'retrolisthesis',r'listesis'],
'disc_herniation':[r'herniation',r'hernia\s+discal'],
'disc_bulging':[r'bulging',r'disc\s+bulge',r'abombamiento']}
CP={k:[re.compile(x,re.I) for x in v] for k,v in PAT.items()}
allowed={'table','structured','text','xml'}; counts=Counter(); scanned=0; schema=[]
for _,r in private.iterrows():
    p=r['_path']; txt=norm(p.name)
    for f,ps in CP.items():
        if any(q.search(txt) for q in ps): counts[(f,'file_name',r.rootAlias)]+=1
    if r.fileGroup not in allowed or p.stat().st_size>MAX_BYTES: continue
    try:
        if r.suffix in {'.csv','.tsv'}:
            df=pd.read_csv(p,sep='\t' if r.suffix=='.tsv' else ',',nrows=MAX_ROWS,dtype=str,low_memory=False); fields=list(df.columns); content=' '.join(df.fillna('').astype(str).to_numpy().ravel())
        elif r.suffix in {'.xlsx','.xls'}:
            wb=pd.ExcelFile(p); fields=[]; vals=[]
            for sh in wb.sheet_names[:10]:
                df=pd.read_excel(p,sheet_name=sh,nrows=MAX_ROWS,dtype=str); fields += [f'{sh}:{c}' for c in df.columns]; vals += df.fillna('').astype(str).to_numpy().ravel().tolist()
            content=' '.join(vals)
        else:
            content=p.read_text(encoding='utf-8',errors='replace'); fields=[]
            if r.suffix=='.json':
                obj=json.loads(content); fields=list(obj.keys()) if isinstance(obj,dict) else (list(obj[0].keys()) if isinstance(obj,list) and obj and isinstance(obj[0],dict) else [])
            elif r.suffix=='.xml':
                fields=[]
                for _,e in ET.iterparse(p,events=('start',)):
                    fields.append(e.tag.split('}')[-1])
                    if len(fields)>=500: break
        for x in set(map(str,fields)): schema.append({'rootAlias':r.rootAlias,'relativePathHash':r.relativePathHash,'fieldName':x[:200],'normalizedFieldName':norm(x)[:200]})
        scanned+=1
        if scanned<=MAX_TEXT:
            for f,ps in CP.items():
                if any(q.search(content) for q in ps): counts[(f,'bounded_content_scan',r.rootAlias)]+=1
    except Exception: pass
schema_df=pd.DataFrame(schema,columns=['rootAlias','relativePathHash','fieldName','normalizedFieldName'])
evidence=pd.DataFrame([{'findingType':f,'evidenceType':t,'rootAlias':r,'sourceCount':c} for (f,t,r),c in sorted(counts.items())],columns=['findingType','evidenceType','rootAlias','sourceCount'])
pre=[]
for f in PAT:
    s=evidence[evidence.findingType==f]; types=sorted(s.evidenceType.unique()) if not s.empty else []; n=int(s.sourceCount.sum()) if not s.empty else 0
    status='NO_EVIDENCE_FOUND' if n==0 else ('TEXT_OR_TABULAR_EVIDENCE_FOUND_REQUIRES_ALIGNMENT_AUDIT' if 'bounded_content_scan' in types else 'STRUCTURAL_HINT_ONLY')
    pre.append({'findingType':f,'evidenceSourceCount':n,'evidenceTypes':'|'.join(types),'precheckStatus':status,'trainingAuthorized':False,'groundTruthCreated':False,'imageAlignmentValidated':False})
pre_df=pd.DataFrame(pre); display(pre_df)

,findingType,evidenceSourceCount,evidenceTypes,precheckStatus,trainingAuthorized,groundTruthCreated,imageAlignmentValidated
0,facet_hypertrophy,0,,NO_EVIDENCE_FOUND,False,False,False
1,ligamentum_flavum_hypertrophy,0,,NO_EVIDENCE_FOUND,False,False,False
2,annular_tear,0,,NO_EVIDENCE_FOUND,False,False,False
3,nerve_root_compression,0,,NO_EVIDENCE_FOUND,False,False,False
4,epidural_fat,0,,NO_EVIDENCE_FOUND,False,False,False
5,disc_height,0,,NO_EVIDENCE_FOUND,False,False,False
6,ap_diameter,0,,NO_EVIDENCE_FOUND,False,False,False
7,spondylolisthesis,0,,NO_EVIDENCE_FOUND,False,False,False
8,disc_herniation,0,,NO_EVIDENCE_FOUND,False,False,False
9,disc_bulging,0,,NO_EVIDENCE_FOUND,False,False,False


In [16]:
OUT.mkdir(parents=True,exist_ok=True)
paths={'rootRegistry':OUT/'dataset_root_candidates_v1.csv','fileSummary':OUT/'file_inventory_summary_v1.csv','schema':OUT/'schema_field_inventory_v1.csv','evidence':OUT/'finding_keyword_evidence_v1.csv','precheck':OUT/'candidate_finding_precheck_v1.csv'}
root_df.to_csv(paths['rootRegistry'],index=False); summary.to_csv(paths['fileSummary'],index=False); schema_df.to_csv(paths['schema'],index=False); evidence.to_csv(paths['evidence'],index=False); pre_df.to_csv(paths['precheck'],index=False)
resolved=len(root_df)>0; status='NOTEBOOK_71_COMPLETE' if resolved else 'NOTEBOOK_71_BLOCKED_DATA_ROOT_NOT_RESOLVED'
marker={'schemaVersion':'pfi.p10-8.notebook-71-complete.v1','status':status,'generatedAtUtc':datetime.now(timezone.utc).isoformat(),'trainingExecuted':False,'weightsDeserialized':False,'internalTestAccessed':False,'officialHiddenTestAccessed':False,'patientIdentifiersExported':False,'clinicalGroundTruthCreated':False,'trainingAuthorized':False,'dataRootResolved':resolved,'rootCount':int(len(root_df)),'indexedFileCount':int(len(private)),'inventoryTruncated':bool(truncated),'boundedTextSourcesScanned':int(scanned),'findingEvidenceRows':int(len(evidence)),'outputs':{k:str(v) for k,v in paths.items()}}
write_json(OUT/'NOTEBOOK_71_COMPLETE.json',marker); print(json.dumps(marker,indent=2,ensure_ascii=False)); print(status)
if not resolved: print('Definí PFI_SUDIRMAN_ROOT, PFI_ALKAFRI_ROOT o PFI_P10_8_DATA_ROOTS y repetí.')

{
  "schemaVersion": "pfi.p10-8.notebook-71-complete.v1",
  "status": "NOTEBOOK_71_COMPLETE",
  "generatedAtUtc": "2026-08-07T01:14:36.180815+00:00",
  "trainingExecuted": false,
  "weightsDeserialized": false,
  "internalTestAccessed": false,
  "officialHiddenTestAccessed": false,
  "patientIdentifiersExported": false,
  "clinicalGroundTruthCreated": false,
  "trainingAuthorized": false,
  "dataRootResolved": true,
  "rootCount": 2,
  "indexedFileCount": 54593,
  "inventoryTruncated": false,
  "boundedTextSourcesScanned": 15,
  "findingEvidenceRows": 0,
  "outputs": {
    "rootRegistry": "/content/drive/MyDrive/PFI_MVP/results/P10_8_clinical_expansion_preflight/sudirman_alkafri_audit_v2/dataset_root_candidates_v1.csv",
    "fileSummary": "/content/drive/MyDrive/PFI_MVP/results/P10_8_clinical_expansion_preflight/sudirman_alkafri_audit_v2/file_inventory_summary_v1.csv",
    "schema": "/content/drive/MyDrive/PFI_MVP/results/P10_8_clinical_expansion_preflight/sudirman_alkafri_audit_v2/sch

In [18]:
from pathlib import Path
import json
import re

import pandas as pd

RESULT_ROOT = Path(
    "/content/drive/MyDrive/PFI_MVP/results/"
    "E7_alkafri_axial_curated_subset"
)

KEYWORDS = re.compile(
    r"(path|root|dir|directory|source|url|dataset|archive|download|"
    r"image|ground.?truth|mask|annotation)",
    re.IGNORECASE,
)

found = []


def inspect_json(value, location="root"):
    if isinstance(value, dict):
        for key, child in value.items():
            child_location = f"{location}.{key}"

            if KEYWORDS.search(str(key)) and isinstance(
                child, (str, int, float, bool)
            ):
                found.append(
                    {
                        "file": current_file.name,
                        "location": child_location,
                        "value": str(child)[:1000],
                    }
                )

            inspect_json(child, child_location)

    elif isinstance(value, list):
        for index, child in enumerate(value[:100]):
            inspect_json(child, f"{location}[{index}]")


for current_file in sorted(RESULT_ROOT.glob("*")):
    try:
        if current_file.suffix.lower() == ".json":
            payload = json.loads(
                current_file.read_text(
                    encoding="utf-8",
                    errors="replace",
                )
            )
            inspect_json(payload)

        elif current_file.suffix.lower() == ".csv":
            frame = pd.read_csv(
                current_file,
                nrows=100,
                dtype=str,
                low_memory=False,
            )

            candidate_columns = [
                column
                for column in frame.columns
                if KEYWORDS.search(str(column))
            ]

            for column in candidate_columns:
                samples = (
                    frame[column]
                    .dropna()
                    .astype(str)
                    .drop_duplicates()
                    .head(10)
                    .tolist()
                )

                for sample in samples:
                    found.append(
                        {
                            "file": current_file.name,
                            "location": f"column:{column}",
                            "value": sample[:1000],
                        }
                    )

    except Exception as exc:
        print(
            "No se pudo inspeccionar:",
            current_file.name,
            type(exc).__name__,
        )

print("Referencias encontradas:", len(found))

for row in found:
    print()
    print("Archivo:", row["file"])
    print("Campo:", row["location"])
    print("Valor:", row["value"])

Referencias encontradas: 240

Archivo: E7_alkafri_axial_candidate_sanity_checks_DECODED.csv
Campo: column:image_file_path
Valor: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_0005/T1_TSE_TRA__0001_006.ima

Archivo: E7_alkafri_axial_candidate_sanity_checks_DECODED.csv
Campo: column:image_file_path
Valor: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_0005/T1_TSE_TRA__0001_003.ima

Archivo: E7_alkafri_axial_candidate_sanity_checks_DECODED.csv
Campo: column:image_file_path
Valor: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0001/L-SPINE_LSS_20160309_091629_240000/T1_TSE_TRA_0005/T1_TSE_TRA__0001_005.ima

Archivo: E7_alkafri_axial_candidate_sanity_checks_DECODED.csv
Campo: column:image_file_path
Valor: /content/drive/MyDr

In [19]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI")

MRI_ROOT = (
    BASE
    / "extracted/_nested/main_dataset__MRI_Data/01_MRI_Data"
)

GT_ROOT = (
    BASE
    / "extracted/_nested/ground_truth__Ground_Truth_Label/"
      "04_Intermediary_Ground_Truth_Data"
)

MANUAL_GT_ROOT = (
    BASE
    / "extracted/_nested/ground_truth__Manual_Label_Data/"
      "03_Manual_Label_Data"
)

paths = {
    "AXIAL_ALKAFRI": BASE,
    "MRI originales": MRI_ROOT,
    "Ground truth procesado": GT_ROOT,
    "Ground truth manual": MANUAL_GT_ROOT,
}

print("=== EXISTENCIA DE CARPETAS ===")

for name, path in paths.items():
    print()
    print(name)
    print("Ruta:", path)
    print("Existe:", path.exists())
    print("Es carpeta:", path.is_dir())

print()
print("=== ARCHIVOS DE REFERENCIA ===")

result_root = Path(
    "/content/drive/MyDrive/PFI_MVP/results/"
    "E7_alkafri_axial_curated_subset"
)

index_path = result_root / "E7_alkafri_axial_image_case_index.csv"
gt_index_path = result_root / "E7_alkafri_gt_case_index.csv"

if index_path.is_file():
    image_index = pd.read_csv(index_path, dtype=str, low_memory=False)

    for value in (
        image_index["image_file_path"]
        .dropna()
        .drop_duplicates()
        .head(5)
    ):
        path = Path(value)
        print("Imagen:", path.exists(), path)

if gt_index_path.is_file():
    gt_index = pd.read_csv(gt_index_path, dtype=str, low_memory=False)

    for value in (
        gt_index["gt_file_path"]
        .dropna()
        .drop_duplicates()
        .head(5)
    ):
        path = Path(value)
        print("Ground truth:", path.exists(), path)

print()
print("=== POSIBLES ARCHIVOS COMPRIMIDOS ===")

archive_patterns = [
    "*.zip",
    "*.tar",
    "*.tar.gz",
    "*.tgz",
    "*.7z",
    "*.rar",
]

archives = []

for search_root in [
    BASE,
    Path("/content/drive/MyDrive/PFI_MVP/data"),
]:
    if not search_root.exists():
        continue

    for pattern in archive_patterns:
        archives.extend(search_root.glob(pattern))

if archives:
    for archive in sorted(set(archives)):
        print(archive)
else:
    print("No se encontraron archivos comprimidos en las carpetas inmediatas.")

=== EXISTENCIA DE CARPETAS ===

AXIAL_ALKAFRI
Ruta: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI
Existe: True
Es carpeta: True

MRI originales
Ruta: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data
Existe: True
Es carpeta: True

Ground truth procesado
Ruta: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data
Existe: True
Es carpeta: True

Ground truth manual
Ruta: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data
Existe: True
Es carpeta: True

=== ARCHIVOS DE REFERENCIA ===
Imagen: True /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_384_0004/T2_TSE_TRA__0037_001.ima
Imagen: True /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_

## Interpretación

`TEXT_OR_TABULAR_EVIDENCE_FOUND_REQUIRES_ALIGNMENT_AUDIT` y `STRUCTURAL_HINT_ONLY` **no habilitan entrenamiento**. Notebook 72 solo podrá estructurar etiquetas si existe asociación inequívoca con paciente, serie, nivel, lado y corte/bloque.